# Lab 2: Pauli Group and Clifford Group in stim

**[EQE5006] Quantum Error Correction, Spring 2026**

In this lab, you will explore the algebraic structure of the **Pauli group** and the **Clifford group** using stim's `Tableau` and `TableauSimulator` APIs. Starting with a review of Lab 1 concepts, you will verify the conjugation tables from Lecture 03, investigate the structure of the single-qubit Clifford group, and experience the practical implications of the Gottesman-Knill theorem.

**Prerequisites**: Lab 1 (Stim Basics), Lecture 03 (Pauli, Clifford, and Universality)

In [ ]:
# Install stim if not already installed
!pip install stim -q

In [ ]:
import stim
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"stim version: {stim.__version__}")

---

## Section 0: Lab 1 Review

Let's warm up by revisiting the core stim workflow from Lab 1: build a circuit with `stim.Circuit()` and `.append()`, then sample measurement outcomes with `.compile_sampler().sample()`. We will also see how Hadamard conjugation transforms noise channels — a direct computational verification of Lecture 03 Exercise 2.

### Warm-up: gate combination $H \to S \to S \to H$

Recall from Lecture 01 that $S^2 = Z$ and $HZH = X$. Combining these:

$$H \to S \to S \to H = H S^2 H = HZH = X$$

So the circuit applies $X$ to $|0\rangle$, giving $|1\rangle$ with certainty.

In [ ]:
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("S", [0])
circuit.append("S", [0])
circuit.append("H", [0])
circuit.append("M", [0])

samples = circuit.compile_sampler().sample(shots=10000)
print(f"P(1) = {np.mean(samples[:, 0]):.4f}  (theory: 1.0000)")

### Hadamard conjugation of noise channels

In Lab 1, we simulated the bit-flip channel $\mathcal{N}_X(\rho) = (1-p)\rho + pX\rho X$ using `X_ERROR(p)`. From Lecture 03, we know that $HXH = Z$. This means that sandwiching a bit-flip error between two Hadamard gates produces a **phase-flip error**:

$$H \circ \mathcal{N}_X \circ H = \mathcal{N}_Z$$

Let's see what this looks like in practice. First, recall the direct bit-flip error on $|0\rangle$:

In [ ]:
# Direct bit-flip error on |0>
p = 0.1
shots = 100000

circuit = stim.Circuit()
circuit.append("X_ERROR", [0], p)
circuit.append("M", [0])
samples = circuit.compile_sampler().sample(shots=shots)
print(f"X_ERROR({p}) on |0>: error rate = {np.mean(samples[:, 0]):.4f}  (theory: {p})")

In [ ]:
# Phase-flip error (H X_ERROR H) on |0>
circuit = stim.Circuit()
circuit.append("H", [0])
circuit.append("X_ERROR", [0], p)
circuit.append("H", [0])
circuit.append("M", [0])
samples = circuit.compile_sampler().sample(shots=shots)
print(
    f"H X_ERROR({p}) H on |0>: error rate = {np.mean(samples[:, 0]):.4f}  (theory: 0)"
)

The error rate dropped to zero! The circuit `H -> X_ERROR(p) -> H` effectively applies `Z_ERROR(p)`. Since $Z|0\rangle = |0\rangle$ ($|0\rangle$ is an eigenstate of $Z$ with eigenvalue $+1$), the phase-flip error is completely invisible when measuring in the $Z$-basis.

### Exercise 0.1

Now consider the reverse transformation. Since $HZH = X$, sandwiching `Z_ERROR(p)` between Hadamard gates should produce a bit-flip error.

**(a)** Predict the error rate of `Z_ERROR(p) -> M` on $|0\rangle$. Verify for $p \in \{0.01, 0.05, 0.1, 0.2, 0.5\}$.

**(b)** Predict the error rate of `H -> Z_ERROR(p) -> H -> M` on $|0\rangle$. Verify for the same $p$ values.

**Question**: Compare your results with the bit-flip channel from Lab 1 (Exercise 3.1). What pattern do you observe?

In [ ]:
# Exercise 0.2(a): Z_ERROR(p) on |0>
# Your prediction: error rate = ???

p_values = [0.01, 0.05, 0.1, 0.2, 0.5]
shots = 100000

print("(a) Z_ERROR(p) on |0>:")
for p in p_values:
    # Your code here
    pass

In [ ]:
# Exercise 0.2(b): H -> Z_ERROR(p) -> H -> M on |0>
# Your prediction: error rate = ???

print("(b) H Z_ERROR(p) H on |0>:")
for p in p_values:
    # Your code here
    pass

---

## Section 1: Pauli Conjugation with `stim.Tableau`

In Lecture 03, we saw that Clifford gates are defined by how they **conjugate** Pauli operators: $P \mapsto UPU^\dagger$. stim provides a dedicated class, `stim.Tableau`, that represents exactly this conjugation action.

A `stim.Tableau` object encodes how a Clifford gate transforms each single-qubit $X$ and $Z$ generator under conjugation. Since $Y = iXZ$, the mapping of $Y$ is determined automatically from $X$ and $Z$.

### The Hadamard tableau

Let's start by inspecting the Hadamard gate's conjugation action.

In [ ]:
t_h = stim.Tableau.from_named_gate("H")
print("H tableau:")
print(t_h)

The printed format is compact:
- Header `+-xz-` labels two columns: `x` (what $X$ maps to) and `z` (what $Z$ maps to).
- Sign row `| ++` shows the phase of each output ($+$ or $-$).
- Pauli row `| ZX` shows the output Pauli: column `x` gives $Z$ (meaning $X \to Z$), column `z` gives $X$ (meaning $Z \to X$).

A more readable way to inspect conjugation is the **call syntax** with `stim.PauliString`. Calling a tableau on a Pauli string returns the conjugated result $UPU^\dagger$:

In [ ]:
print("H conjugation:")
print(f"  X -> {t_h(stim.PauliString('X'))}")
print(f"  Z -> {t_h(stim.PauliString('Z'))}")
print(f"  Y -> {t_h(stim.PauliString('Y'))}")

This matches the conjugation table from Lecture 03: $H$ swaps $X \leftrightarrow Z$ and maps $Y \to -Y$.

In [ ]:
# S gate
t_s = stim.Tableau.from_named_gate("S")
print("S conjugation:")
print(f"  X -> {t_s(stim.PauliString('X'))}")
print(f"  Z -> {t_s(stim.PauliString('Z'))}")
print(f"  Y -> {t_s(stim.PauliString('Y'))}")

### Multi-qubit tableaux

For multi-qubit gates, the PauliString call syntax is especially useful. Use `_` to denote the identity on a qubit (e.g., `"X_"` means $X \otimes I$, `"_Z"` means $I \otimes Z$).

In [ ]:
t_cnot = stim.Tableau.from_named_gate("CNOT")

print("CNOT conjugation (control=q0, target=q1):")
print(f"  X_0 (X_) -> {t_cnot(stim.PauliString('X_'))}")
print(f"  Z_0 (Z_) -> {t_cnot(stim.PauliString('Z_'))}")
print(f"  X_1 (_X) -> {t_cnot(stim.PauliString('_X'))}")
print(f"  Z_1 (_Z) -> {t_cnot(stim.PauliString('_Z'))}")

Compare with the Lecture 03 conjugation table:

- $X$ spreads **forward** (control to target): $X_0 \to X_0 X_1$
- $Z$ spreads **backward** (target to control): $Z_1 \to Z_0 Z_1$
- $Z_0$ and $X_1$ are unchanged

### Tableau composition

Tableaux can be multiplied to represent the composition of Clifford operations. In stim, `t_a * t_b` means "apply gate `b` first, then gate `a`" — this follows the standard convention for operator composition ($AB$ means apply $B$ first).

For example, $H^2 = I$:

In [ ]:
print(t_h * t_h)
identity = stim.Tableau(1)
print(t_h * t_h == identity)

In [ ]:
# Composing S then H (apply S first, then H):
t_hs = t_h * t_s
print(f"\nCircuit S -> H (= unitary HS):")
print(f"  X -> {t_hs(stim.PauliString('X'))}")
print(f"  Z -> {t_hs(stim.PauliString('Z'))}")

To build the tableau for a longer circuit $G_1 \to G_2 \to \cdots \to G_n$ (gates applied in time order), you can left-multiply step by step:

```python
combined = stim.Tableau(1)  # identity
for gate_tableau in [t_g1, t_g2, ..., t_gn]:
    combined = gate_tableau * combined
```

### Exercise 1.1 — Pauli gate conjugation tables

For each Pauli gate ($X$, $Y$, $Z$), create its tableau and print the conjugation of $X$ and $Z$.

**Question**: Verify that every Pauli gate maps Pauli operators to Pauli operators (possibly with a sign change). Why does this confirm that Pauli gates are elements of the Clifford group?

In [ ]:
# Exercise 1.1: Pauli gate conjugation tables

for gate_name in ["X", "Y", "Z"]:
    # Your code here
    pass

### Exercise 1.2 — Composite conjugation

For each of the following circuits, **predict** the conjugation table ($X \to ?,\; Z \to ?$) by hand using the individual tables from Lecture 03, then **verify** with stim.

**(a)** $S \to H$ (apply $S$ first, then $H$)

**(b)** $H \to S \to H$

In [ ]:
# Exercise 1.2(a): Circuit S -> H
# Your prediction: X -> ???, Z -> ???

# Your code here

In [ ]:
# Exercise 1.2(b): Circuit H -> S -> H
# Your prediction: X -> ???, Z -> ???

# Your code here

---

## Section 2: Clifford Group Structure

Now that we can compute tableaux, let's explore the algebraic structure of the Clifford group itself. In Lecture 03, we learned that:

- The Clifford group $\mathcal{C}_n$ is generated by $\{H, S, \text{CNOT}\}$
- The single-qubit Clifford group has $|\mathcal{C}_1| = 24$ elements
- The Pauli group is a subgroup: $\mathcal{P}_n \leq \mathcal{C}_n$

Let's verify these facts computationally.

### Tableau equality and identity

Two tableaux can be compared with `==`, and `stim.Tableau(n)` creates the $n$-qubit identity.

In [ ]:
identity = stim.Tableau(1)
t_h = stim.Tableau.from_named_gate("H")

# H^2 = I
t_h * t_h == identity

In [ ]:
# S^2 = Z
t_s = stim.Tableau.from_named_gate("S")
t_z = stim.Tableau.from_named_gate("Z")
t_s**2 == t_z

In [ ]:
# S^4 = I (since S^2 = Z and Z^2 = I)
t_s**4 == identity

### Counting $|\mathcal{C}_1| = 24$

In Lecture 03, we stated that the single-qubit Clifford group has 24 elements generated by $H$ and $S$. Let's verify this by **breadth-first search**: starting from the identity, we repeatedly left-multiply by $H$ and $S$, collecting all distinct tableaux until no new ones appear.

In [ ]:
t_h = stim.Tableau.from_named_gate("H")
t_s = stim.Tableau.from_named_gate("S")
identity = stim.Tableau(1)

# BFS to find all distinct single-qubit Clifford operations
seen_strs = {str(identity)}
queue = [identity]

while queue:
    current = queue.pop(0)
    for gate in [t_h, t_s]:
        new = gate * current
        key = str(new)
        if key not in seen_strs:
            seen_strs.add(key)
            queue.append(new)
            print(key)
            print()

In [ ]:
len(seen_strs)

### Exercise 2.1 — Mystery circuit

A circuit applies gates in order: $S \to S \to H \to S \to S \to H$.

Compute its combined tableau and determine which **single gate** has the same conjugation action.

*Hint*: Use the step-by-step composition pattern shown earlier. Then compare the result against known gate tableaux.

In [ ]:
# Exercise 2.2: Mystery circuit S -> S -> H -> S -> S -> H

t_h = stim.Tableau.from_named_gate("H")
t_s = stim.Tableau.from_named_gate("S")

# TODO: Compute the combined tableau for the circuit

# Your code here

# TODO: Compare with known gates to identify the equivalent operation

# Your code here

---

## Section 3: Gottesman-Knill in Practice

In Lecture 03, we stated the **Gottesman-Knill theorem**: Clifford circuits with $Z$ initializations and measurements can be efficiently simulated on a classical computer. The key insight is that an $n$-qubit Clifford operation can be represented by a **tableau** of size $O(n^2)$, rather than a state vector of size $2^n$.

stim leverages this. In this section, we introduce `stim.TableauSimulator` for interactive Clifford simulation, and then benchmark stim's performance to see the Gottesman-Knill theorem in action.

### Introducing `stim.TableauSimulator`

`stim.TableauSimulator` is an interactive simulator that maintains the quantum state internally using the tableau representation. You can apply gates one at a time and perform measurements.

In [ ]:
# Create a simulator and build a Bell state
sim = stim.TableauSimulator()
sim.h(0)
sim.cnot(0, 1)

# Measure both qubits
m0 = sim.measure(0)
m1 = sim.measure(1)
print(f"Qubit 0: {int(m0)}, Qubit 1: {int(m1)}")
print(f"Qubits agree: {m0 == m1}")

You can also apply an entire `stim.Circuit` at once using `do_circuit()`:

In [ ]:
# Applying a circuit to the simulator
sim = stim.TableauSimulator()

bell_circuit = stim.Circuit()
bell_circuit.append("H", [0])
bell_circuit.append("CNOT", [0, 1])

sim.do_circuit(bell_circuit)

m0 = sim.measure(0)
m1 = sim.measure(1)
print(f"Qubit 0: {int(m0)}, Qubit 1: {int(m1)}, Agree: {m0 == m1}")

### Exercise 3.1 — Benchmarking Clifford simulation

How does stim's simulation time scale with the number of qubits? Let's find out by timing the compilation and sampling of random Clifford circuits.

The function below generates a random Clifford circuit with `n_qubits` qubits and `depth` layers. Each layer applies a random single-qubit Clifford gate ($H$, $S$, or $S^\dagger$) to every qubit, followed by CNOT gates between adjacent pairs.

In [ ]:
def make_random_clifford_circuit(n_qubits, depth, rng):
    """Create a random Clifford circuit on n_qubits with given depth."""
    circuit = stim.Circuit()
    gates_1q = ["H", "S", "S_DAG"]
    for d in range(depth):
        # Single-qubit layer
        for q in range(n_qubits):
            gate = rng.choice(gates_1q)
            circuit.append(gate, [q])
        # Two-qubit layer: CNOT on adjacent pairs (alternating even/odd)
        start = d % 2
        for q in range(start, n_qubits - 1, 2):
            circuit.append("CNOT", [q, q + 1])
    circuit.append("M", list(range(n_qubits)))
    return circuit


# Benchmark: vary qubit count, fixed depth
n_values = np.arange(20, 101, 20)
depth = 100
times = []
rng = np.random.default_rng(42)

for n in n_values:
    circuit = make_random_clifford_circuit(n, depth, rng)
    start = time.perf_counter()
    circuit.compile_sampler().sample(shots=1)
    elapsed = time.perf_counter() - start
    times.append(elapsed)
    print(f"n = {n:>5d} qubits: {elapsed:.4f} s")

In [ ]:
# --- Plotting template ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(n_values, times, "bo-", markersize=8, label="Measured")

# Linear fit in log-log space to determine scaling exponent
log_n = np.log(n_values)
log_t = np.log(times)
coeffs = np.polyfit(log_n, log_t, 1)
slope, intercept = coeffs
print(f"Scaling exponent (gradient): {slope:.2f}")

# Plot the fit line
fit_times = np.exp(intercept) * n_values**slope
ax.loglog(
    n_values, fit_times, "b--", alpha=0.7, label=f"Fit: $t \\propto n^{{{slope:.2f}}}$"
)

ax.set_xlabel("Number of qubits")
ax.set_ylabel("Simulation time (s)")
ax.set_title("Clifford circuit simulation: time vs qubit count (depth=100)")
ax.grid(True, alpha=0.3, which="both")
ax.legend()
plt.tight_layout()
plt.show()

**Question**: What is the approximate scaling of simulation time with qubit count? If you were using a state vector simulator instead, how would the memory requirement grow with $n$? (Consider that an $n$-qubit state vector has $2^n$ complex amplitudes, each requiring 16 bytes.)

### Why Clifford circuits are classically simulable

The benchmark above demonstrates the Gottesman-Knill theorem in practice:

- stim can simulate **10,000+ qubit** Clifford circuits in seconds
- A state vector simulator would need $2^n$ complex amplitudes — for $n = 50$, that alone requires ~8 PB of memory
- The tableau representation uses only $O(n^2)$ bits to track how each Pauli generator transforms
- Clifford circuits can produce highly entangled states (e.g., GHZ states on thousands of qubits), yet they remain classically simulable

This means **entanglement alone does not guarantee quantum computational advantage**. To go beyond classical simulation, we need **non-Clifford gates** like the $T$ gate, which breaks out of the Pauli-preserving structure that makes efficient tableau tracking possible.

### Exercise 3.2 — Depth scaling

Fix the number of qubits at $n = 100$ and vary the circuit depth. How does simulation time scale with depth?

**Question**: Compare the depth scaling with the qubit-count scaling from Exercise 3.1. Which dimension is more expensive to increase?

In [ ]:
# Exercise 3.2: Depth scaling (n = 100, varying depth)

# TODO: Build a random circuit with n = 100 qubits while varying depths

---

## Bonus Exercises (Optional)

The following exercises preview the **stabilizer formalism** and **quantum error correction**, which will be covered in upcoming lectures. They are optional challenges for those who finish early.

### Bonus 1 — Stabilizers of the Bell state

The state $|00\rangle$ is a $+1$ eigenstate of both $Z_0$ and $Z_1$. We say $Z_0$ and $Z_1$ **stabilize** $|00\rangle$.

When we apply an encoding circuit $U$ to $|00\rangle$, the stabilizers transform accordingly: if $Z_0 |00\rangle = |00\rangle$, then

$$(U Z_0 U^\dagger)(U|00\rangle) = U Z_0 |00\rangle = U|00\rangle$$

So $U Z_0 U^\dagger$ stabilizes the encoded state $U|00\rangle$. Use Tableau to find the stabilizers of the Bell state $|\Phi^+\rangle = \text{CNOT}_{01}(H_0 \otimes I)|00\rangle$.

*Hint*: The `+` operator on tableaux computes the **direct sum** (tensor product of independent operations). So `stim.Tableau.from_named_gate("H") + stim.Tableau(1)` gives $H \otimes I$ on 2 qubits.

**Questions**:
- What are the two stabilizers of $|\Phi^+\rangle$?
- How does this relate to Lecture 03 Exercise 1?

In [ ]:
# Bonus 1: Stabilizers of the Bell state

# Step 1: Build the 2-qubit encoding tableau (H on q0, then CNOT)
t_h_2q = stim.Tableau.from_named_gate("H") + stim.Tableau(1)  # H ⊗ I
# TODO: Your code here

# Step 2: Track how the initial stabilizers Z_0 and Z_1 transform
# TODO: Your code here